## Task 1: BFS for Minimum-Transfer Public Transport Routing

In [ ]:
from collections import deque
import networkx as nx
import matplotlib.pyplot as plt

# ---------------------------------------------------------------------------
# 1. Transportation Network – complete adjacency list
#    Directed graph: each entry lists stations reachable in one transition
# ---------------------------------------------------------------------------
graph = {
    "FAST University": [
        "NUST Metro Station"
    ],
    "NUST Metro Station": [
        "Saddar Metro Station",
        "Faiz Ahmed Faiz Metro Station",
        "G-11 Metro Station"
    ],
    "Saddar Metro Station": [
        "Faiz Ahmed Faiz Metro Station",
        "Zero Point Metro Station"
    ],
    "Faiz Ahmed Faiz Metro Station": [
        "PIMS Metro Station"
    ],
    "G-11 Metro Station": [],
    "Zero Point Metro Station": [
        "PIMS Metro Station"
    ],
    "PIMS Metro Station": []
}

# ---------------------------------------------------------------------------
# 2. BFS Function
#
#  Key design decisions (per lab spec):
#   - FIFO queue (deque) as the frontier
#   - Start node marked visited immediately to prevent re-queuing
#   - Goal test applied when node is *removed* from the queue
#   - parent dict enables path reconstruction without storing full paths
#   - level dict records BFS depth (= min transitions from start)
# ---------------------------------------------------------------------------
def bfs(graph, start, goal):
    queue   = deque([start])   # FIFO frontier
    visited = {start}          # mark start immediately so it is never re-added
    parent  = {start: None}    # parent[v] = u means u was expanded to discover v
    level   = {start: 0}       # BFS level == number of transitions from start
    expansion_order = []

    while queue:
        print(f"  Frontier: {list(queue)}")
        current = queue.popleft()          # FIFO: oldest node first
        expansion_order.append(current)

        # Goal test when node is removed from the queue
        if current == goal:
            break

        for neighbor in graph[current]:
            if neighbor not in visited:
                visited.add(neighbor)
                parent[neighbor] = current
                level[neighbor]  = level[current] + 1
                queue.append(neighbor)

    # Reconstruct solution path by following parent pointers back from goal
    path = []
    node = goal
    while node is not None:
        path.append(node)
        node = parent.get(node)
    path.reverse()

    return expansion_order, level, path


# ---------------------------------------------------------------------------
# 3. Run BFS
# ---------------------------------------------------------------------------
start = "FAST University"
goal  = "PIMS Metro Station"

print("=== BFS TRAVERSAL LOG ===")
order, levels, path = bfs(graph, start, goal)

# ---------------------------------------------------------------------------
# 4. Display Results
# ---------------------------------------------------------------------------
print("\n====== RESULTS OF BFS SEARCH ======")
print("Initial State:", start)
print("Goal State   :", goal)

print("\nLevels (along solution path):")
for node in path:
    print(f"  Level {levels[node]}: {node}")

print("\nTraversal / Expansion Order:")
for node in order:
    print(f"  {node}")

print("\nSolution Path:")
print("  " + " \u2192 ".join(path))

print("\nNumber of Transitions:", len(path) - 1)

# ---------------------------------------------------------------------------
# 5. NetworkX Visualization
#
#  Color scheme:
#   - limegreen  = initial state
#   - tomato     = goal state
#   - gold        = intermediate nodes on solution path
#   - lightblue  = explored but not on path
#
#  Solution path edges are drawn in red with higher line width.
#  Key transport service labels annotate the main route edges.
# ---------------------------------------------------------------------------
G = nx.DiGraph()
for node in graph:
    for neighbor in graph[node]:
        G.add_edge(node, neighbor)

# Fixed positions that approximate the real Islamabad transport layout
pos = {
    "FAST University":               (-3.0,  0.0),
    "NUST Metro Station":            (-1.5,  0.0),
    "Saddar Metro Station":          (-0.3,  1.4),
    "Zero Point Metro Station":      ( 1.2,  1.4),
    "G-11 Metro Station":            (-0.3, -1.4),
    "Faiz Ahmed Faiz Metro Station": ( 0.7,  0.0),
    "PIMS Metro Station":            ( 2.2,  0.0),
}

path_set   = set(path)
path_edges = list(zip(path, path[1:]))

node_colors = []
for node in G.nodes():
    if node == start:
        node_colors.append("limegreen")
    elif node == goal:
        node_colors.append("tomato")
    elif node in path_set:
        node_colors.append("gold")
    else:
        node_colors.append("lightblue")

non_path_edges = [e for e in G.edges() if e not in path_edges]

plt.figure(figsize=(16, 6))
ax = plt.gca()

nx.draw_networkx_nodes(G, pos, node_color=node_colors, node_size=2000, ax=ax)
nx.draw_networkx_labels(G, pos, font_size=6.5, font_weight="bold", ax=ax)

# Non-solution edges in grey
nx.draw_networkx_edges(G, pos, edgelist=non_path_edges,
                       arrows=True, arrowsize=18, edge_color="grey",
                       width=1.5, ax=ax, connectionstyle="arc3,rad=0.08")

# Solution path edges in red
nx.draw_networkx_edges(G, pos, edgelist=path_edges,
                       arrows=True, arrowsize=25, edge_color="red",
                       width=4, ax=ax, connectionstyle="arc3,rad=0.08")

# Transport service labels on key route edges
edge_labels = {
    ("FAST University", "NUST Metro Station"):              "FR-01 (Electric Bus)",
    ("NUST Metro Station", "Faiz Ahmed Faiz Metro Station"): "Orange Line",
    ("Faiz Ahmed Faiz Metro Station", "PIMS Metro Station"): "Red Line",
}
nx.draw_networkx_edge_labels(G, pos, edge_labels=edge_labels,
                              font_size=6, font_color="darkblue", ax=ax)

plt.title("Task 1 \u2013 BFS: Minimum-Transfer Route\n"
          "Green=Start | Red=Goal | Gold=Path | Grey=Other edges", fontsize=11)
plt.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
1+1

In [ ]:
1+1

## Task 2: Tree Search vs Graph Search

In [ ]:
from collections import deque
import networkx as nx
import matplotlib.pyplot as plt

# ===========================================================================
# TASK 2  PART 1 – BFS ON TREE (No Visited Set)
#
# WHY no visited set is safe here:
#   A tree has exactly one path between any two nodes (no back-edges, no
#   cycles). Therefore no node can ever be reached through two different
#   branches, so repeated states are structurally impossible and a visited
#   set is not needed for correctness.
# ===========================================================================
tree = {
    "Main Entrance":        ["Reception", "Emergency Lobby"],
    "Reception":            ["Registration"],
    "Registration":         ["Main Corridor"],
    "Main Corridor":        [],
    "Emergency Lobby":      ["Waiting Area"],
    "Waiting Area":         ["Radiology Department"],
    "Radiology Department": []
}


def bfs_tree(graph, start, goal):
    """
    BFS on a tree without a visited set.

    The parent dict doubles as a 'seen' guard: a neighbor is only added to
    the queue when it has no parent entry yet, preventing any node from
    being queued more than once even without an explicit visited set.
    In a true tree this guard is redundant, but it mirrors good practice.
    """
    queue  = deque([start])
    parent = {start: None}   # No visited set; parent membership = queued
    order  = []
    step   = 1

    while queue:
        current = queue.popleft()
        order.append(current)
        print(f"Step {step}: Take \u2192 {current}")
        step += 1

        # Goal test when removed from queue
        if current == goal:
            print("  >> Goal Found!")
            break

        added = []
        for neighbor in graph[current]:
            if neighbor not in parent:    # only queue if not yet discovered
                parent[neighbor] = current
                queue.append(neighbor)
                added.append(neighbor)

        if added:
            print(f"  Add: {', '.join(added)}")
        print(f"  Queue: {list(queue)}\n")

    # Reconstruct path
    path = []
    node = goal
    while node is not None:
        path.append(node)
        node = parent.get(node)
    path.reverse()

    return order, path


start = "Main Entrance"
goal  = "Radiology Department"

print("=== BFS on TREE (No Visited Set) ===\n")
order, path = bfs_tree(tree, start, goal)

print("\nTraversal Order:")
print("  " + " \u2192 ".join(order))

print("\nSolution Path:")
print("  " + " \u2192 ".join(path))

## Part 2A: BFS on Graph WITHOUT Visited Set

In [ ]:
from collections import deque

# ===========================================================================
# TASK 2  PART 2A – BFS ON GRAPH WITHOUT VISITED SET
#
# The extra edge  Main Corridor <-> Waiting Area  creates a cycle.
# Without a visited set, BFS re-enqueues already-seen nodes indefinitely:
#
#   Main Corridor generates Waiting Area
#   Waiting Area  generates Main Corridor  (again)
#   Main Corridor generates Waiting Area   (again) ... forever
#
# max_steps is a demonstration safety brake, NOT part of BFS logic.
# ===========================================================================
hospital_graph = {
    "Main Entrance":        ["Reception", "Emergency Lobby"],
    "Reception":            ["Registration"],
    "Registration":         ["Main Corridor"],
    "Main Corridor":        ["Waiting Area"],             # back-edge: creates cycle
    "Emergency Lobby":      ["Waiting Area"],
    "Waiting Area":         ["Radiology Department", "Main Corridor"],  # cycle
    "Radiology Department": []
}


def bfs_without_visited(graph, start, goal, max_steps=15):
    """
    Standard BFS with NO visited set.

    On a graph with cycles every already-explored node can be re-added to
    the frontier through a different branch. The search may never terminate.
    max_steps stops the demo; in production this loop would run indefinitely.
    """
    queue = deque([start])
    order = []

    print(f"{'Step':<6} {'Current Node':<26} {'Nodes Added':<36} Queue")
    print("-" * 100)

    step = 1
    while queue and step <= max_steps:
        current = queue.popleft()
        order.append(current)

        if current == goal:
            print(f"{step:<6} {current:<26} {'(GOAL FOUND)':<36} {list(queue)}")
            break

        added = graph[current]
        for neighbor in added:
            queue.append(neighbor)           # no visited check!

        print(f"{step:<6} {current:<26} {str(added):<36} {list(queue)}")
        step += 1
    else:
        if step > max_steps:
            print(f"\n[DEMO STOPPED after {max_steps} steps]")
            print("Observation: 'Main Corridor' and 'Waiting Area' repeatedly")
            print("             re-enter the frontier due to the cycle.")
            print("Without a visited set the search cannot terminate on this graph.")

    return order


order = bfs_without_visited(
    hospital_graph,
    "Main Entrance",
    "Radiology Department"
)

**Observation:** The real problem shown above is that repeated states continue to be generated.
If the goal were unreachable the search would run indefinitely.
The `max_steps` parameter is a *demonstration safety limit only* — it is not part of the BFS concept.

## Part 2B: BFS on Graph WITH Visited Set

In [ ]:
from collections import deque

# ===========================================================================
# TASK 2  PART 2B – BFS ON GRAPH WITH VISITED SET
#
# Adding visited = set() breaks cycles:
#   When Waiting Area tries to re-enqueue Main Corridor the check
#   'Main Corridor in visited' is True  →  SKIP.
#   The frontier never grows unboundedly and search terminates correctly.
# ===========================================================================

def bfs_graph(graph, start, goal):
    """
    BFS with a visited set – the correct approach for cyclic graphs.

    A node is added to visited when it is *first discovered* (before it is
    added to the queue) so it can never be queued twice, preventing cycles.
    """
    queue   = deque([start])
    visited = {start}          # mark on discovery, not on expansion
    parent  = {start: None}
    order   = []

    while queue:
        current = queue.popleft()
        order.append(current)
        print(f"Processing : {current}")
        print(f"  Visited  : {visited}")
        print(f"  Queue    : {list(queue)}\n")

        if current == goal:
            break

        for neighbor in graph[current]:
            if neighbor not in visited:
                visited.add(neighbor)        # mark before queuing
                parent[neighbor] = current
                queue.append(neighbor)

    # Reconstruct path via parent pointers
    path = []
    node = goal
    while node is not None:
        path.append(node)
        node = parent.get(node)
    path.reverse()

    return order, visited, path


order, visited, path = bfs_graph(
    hospital_graph,
    "Main Entrance",
    "Radiology Department"
)

print("\nTraversal Order:")
print("  " + " \u2192 ".join(order))

print("\nVisited Set:")
print(" ", visited)

print("\nSolution Path:")
print("  " + " \u2192 ".join(path))

## NetworkX Visualization

In [ ]:
# ===========================================================================
# TASK 2 VISUALIZATION
#   Shows the hospital graph with the BFS solution path highlighted.
#   The dashed-style cycle edge is annotated to make the structural
#   difference between tree and graph representations visually clear.
# ===========================================================================
G = nx.DiGraph()
for node in hospital_graph:
    for neighbor in hospital_graph[node]:
        G.add_edge(node, neighbor)

pos = {
    "Main Entrance":        ( 0.0,  2.5),
    "Reception":            (-2.0,  0.8),
    "Emergency Lobby":      ( 2.0,  0.8),
    "Registration":         (-2.0, -0.8),
    "Waiting Area":         ( 2.0, -0.8),
    "Main Corridor":        (-2.0, -2.5),
    "Radiology Department": ( 2.0, -2.5),
}

path_set   = set(path)
path_edges = list(zip(path, path[1:]))

node_colors = []
for node in G.nodes():
    if node == "Main Entrance":          node_colors.append("limegreen")
    elif node == "Radiology Department": node_colors.append("tomato")
    elif node in path_set:               node_colors.append("gold")
    else:                                node_colors.append("lightblue")

# Separate cycle edges for distinct styling
cycle_edges    = [("Waiting Area", "Main Corridor"), ("Main Corridor", "Waiting Area")]
regular_edges  = [e for e in G.edges()
                  if e not in path_edges and e not in cycle_edges]

plt.figure(figsize=(10, 9))

nx.draw_networkx_nodes(G, pos, node_color=node_colors, node_size=2400)
nx.draw_networkx_labels(G, pos, font_size=7.5, font_weight="bold")

# Regular (non-path, non-cycle) edges
nx.draw_networkx_edges(G, pos, edgelist=regular_edges,
                       arrows=True, arrowsize=20, edge_color="grey", width=1.5)

# Cycle edges in purple to highlight the structural difference
nx.draw_networkx_edges(G, pos, edgelist=[e for e in cycle_edges if G.has_edge(*e)],
                       arrows=True, arrowsize=20, edge_color="purple",
                       width=2, style="dashed", connectionstyle="arc3,rad=0.25")

# Solution path in red
nx.draw_networkx_edges(G, pos, edgelist=path_edges,
                       arrows=True, arrowsize=25, edge_color="red", width=4)

nx.draw_networkx_edge_labels(
    G, pos,
    edge_labels={("Main Corridor", "Waiting Area"): "CYCLE",
                 ("Waiting Area",  "Main Corridor"): "CYCLE"},
    font_color="purple", font_size=8
)

plt.title(
    "Task 2: BFS Graph Search (With Visited Set)\n"
    "Green=Start | Red=Goal | Gold=Path | Purple dashed=Cycle edges",
    fontsize=10
)
plt.axis("off")
plt.tight_layout()
plt.show()

## Task 3: DFS for Warehouse Robot Exploration

In [ ]:
import networkx as nx
import matplotlib.pyplot as plt

# ===========================================================================
# TASK 3: DFS – WAREHOUSE ROBOT
#
# Graph structure (hierarchical zones):
#   Receiving Area
#     └── Main Aisle
#           ├── Electronics  →  Phones  →  Inspection Area   (GOAL)
#           │                └── Laptops
#           ├── Grocery      →  Food
#           │                └── Drinks
#           └── Packaging    →  Dispatch
# ===========================================================================
warehouse = {
    "Receiving Area":  ["Main Aisle"],
    "Main Aisle":      ["Electronics", "Grocery", "Packaging"],
    "Electronics":     ["Phones", "Laptops"],
    "Phones":          ["Inspection Area"],
    "Laptops":         [],
    "Grocery":         ["Food", "Drinks"],
    "Food":            [],
    "Drinks":          [],
    "Packaging":       ["Dispatch"],
    "Dispatch":        [],
    "Inspection Area": []
}


def dfs(graph, start, goal):
    """
    DFS using an explicit LIFO stack.

    Neighbors are pushed in reversed order so the *first* neighbor in the
    adjacency list is explored first (consistent left-to-right traversal).

    visited check is done *on pop* (not on push) so stale stack entries
    for already-expanded nodes are simply skipped, which correctly handles
    the case where the same neighbor is pushed twice from different parents.

    parent is recorded on push (first discovery) so that the path
    reconstructed is always the first DFS path found to the goal.
    """
    stack          = [start]
    visited        = set()
    parent         = {start: None}
    traversal_order = []

    while stack:
        print(f"  Stack: {stack}")
        current = stack.pop()          # LIFO: most recently added first

        if current in visited:
            continue                   # stale entry – already expanded

        visited.add(current)
        traversal_order.append(current)

        # Goal test when popped from stack
        if current == goal:
            break

        # Push neighbors in reversed order so leftmost is processed first
        for neighbor in reversed(graph[current]):
            if neighbor not in visited:
                stack.append(neighbor)
                if neighbor not in parent:   # record first discovery path
                    parent[neighbor] = current

    # Reconstruct solution path via parent pointers
    path = []
    node = goal
    while node is not None:
        path.append(node)
        node = parent.get(node)
    path.reverse()

    return traversal_order, path, visited


# ---------------------------------------------------------------------------
# Run DFS
# ---------------------------------------------------------------------------
start = "Receiving Area"
goal  = "Inspection Area"

print("=== DFS TRAVERSAL LOG ===")
order, path, visited = dfs(warehouse, start, goal)

print("\n====== RESULTS ======")
print("Traversal Order:")
print("  " + " \u2192 ".join(order))

print("\nSolution Path:")
print("  " + " \u2192 ".join(path))

print("\nNumber of States Explored:", len(visited))

In [ ]:
# ===========================================================================
# TASK 3  NetworkX VISUALIZATION
# ===========================================================================
G = nx.DiGraph()
for node in warehouse:
    for neighbor in warehouse[node]:
        G.add_edge(node, neighbor)

# Hierarchical positions mirroring the warehouse zone diagram
pos = {
    "Receiving Area":  ( 0.0,  4.0),
    "Main Aisle":      ( 0.0,  2.5),
    "Electronics":     (-3.5,  1.0),
    "Grocery":         ( 0.0,  1.0),
    "Packaging":       ( 3.5,  1.0),
    "Phones":          (-4.5, -0.5),
    "Laptops":         (-2.5, -0.5),
    "Food":            (-1.0, -0.5),
    "Drinks":          ( 1.0, -0.5),
    "Dispatch":        ( 3.5, -0.5),
    "Inspection Area": (-4.5, -2.2),
}

path_set   = set(path)
path_edges = list(zip(path, path[1:]))

node_colors = []
for node in G.nodes():
    if node == start:       node_colors.append("limegreen")
    elif node == goal:      node_colors.append("tomato")
    elif node in path_set:  node_colors.append("gold")
    else:                   node_colors.append("lightblue")

non_path_edges = [e for e in G.edges() if e not in path_edges]

plt.figure(figsize=(14, 8))

nx.draw_networkx_nodes(G, pos, node_color=node_colors, node_size=2000)
nx.draw_networkx_labels(G, pos, font_size=7, font_weight="bold")

nx.draw_networkx_edges(G, pos, edgelist=non_path_edges,
                       arrows=True, arrowsize=18, edge_color="grey", width=1.5)

nx.draw_networkx_edges(G, pos, edgelist=path_edges,
                       arrows=True, arrowsize=25, edge_color="red", width=4)

plt.title(
    "Task 3: DFS Warehouse Exploration\n"
    "Green=Start | Red=Goal | Gold=DFS Path | Grey=Other edges",
    fontsize=11
)
plt.axis("off")
plt.tight_layout()
plt.show()

## Task 4: UCS for Delivery Route Optimization

In [ ]:
import heapq
import networkx as nx
import matplotlib.pyplot as plt

# ===========================================================================
# TASK 4: UCS – DELIVERY ROUTE OPTIMIZATION
#
# Weighted delivery network (illustrative laboratory costs):
#
#   Islamabad Delivery Hub
#     ├── I-8 Markaz  (cost 4)  ──► F-8 Markaz (cost 5) ──► Blue Area (cost 3)
#     └── H-8 Markaz  (cost 2)  ──────────────────────────► Blue Area (cost 6)
#
#   Route via I-8 + F-8  : 4 + 5 + 3 = 12   (fewer edges, higher cost)
#   Route via H-8        : 2 + 6     =  8   (more direct, lower cost)
#
#   UCS selects the H-8 route because it has the lower accumulated cost.
#   This demonstrates that UCS optimises cost, NOT number of edges.
# ===========================================================================
graph = {
    "Islamabad Delivery Hub":   [("I-8 Markaz", 4), ("H-8 Markaz", 2)],
    "I-8 Markaz":               [("F-8 Markaz", 5)],
    "H-8 Markaz":               [("Blue Area Delivery Point", 6)],
    "F-8 Markaz":               [("Blue Area Delivery Point", 3)],
    "Blue Area Delivery Point":  []
}


# ---------------------------------------------------------------------------
# 2. UCS FUNCTION
#
#  Priority queue entries: (cumulative_cost, node)
#  heapq is a min-heap, so the lowest-cost node is always expanded next.
#
#  best_cost guards against re-expanding a node via a costlier path that
#  was pushed to the heap before the cheaper path was discovered.
#  This is the standard lazy-deletion pattern for Dijkstra / UCS.
# ---------------------------------------------------------------------------
def ucs(graph, start, goal):
    priority_queue = []
    heapq.heappush(priority_queue, (0, start))   # (cost, node)

    best_cost = {start: 0}    # lowest cost known to reach each node
    parent    = {start: None}
    traversal_order = []

    print(f"{'Expanded':<30} {'Cost':<8} Priority Queue (after pop)")
    print("-" * 80)

    while priority_queue:
        cost, current = heapq.heappop(priority_queue)

        # Lazy-deletion: skip if a cheaper path was already processed
        if cost > best_cost.get(current, float('inf')):
            continue

        traversal_order.append(current)
        print(f"{current:<30} {cost:<8} {priority_queue}")

        # Goal test when node is removed from priority queue
        if current == goal:
            break

        for neighbor, edge_cost in graph[current]:
            new_cost = cost + edge_cost
            if new_cost < best_cost.get(neighbor, float('inf')):
                best_cost[neighbor] = new_cost
                parent[neighbor]    = current
                heapq.heappush(priority_queue, (new_cost, neighbor))

    # Reconstruct solution path
    path = []
    node = goal
    while node is not None:
        path.append(node)
        node = parent.get(node)
    path.reverse()

    return traversal_order, path, best_cost


# ---------------------------------------------------------------------------
# 3. RUN UCS
# ---------------------------------------------------------------------------
start = "Islamabad Delivery Hub"
goal  = "Blue Area Delivery Point"

print("=== UCS TRAVERSAL LOG ===\n")
traversal_order, path, best_cost = ucs(graph, start, goal)

# ---------------------------------------------------------------------------
# 4. DISPLAY RESULTS
# ---------------------------------------------------------------------------
print("\n====== RESULTS ======")
print("Traversal Order:")
print("  " + " \u2192 ".join(traversal_order))

print("\nSolution Path:")
print("  " + " \u2192 ".join(path))

print("\nTotal Path Cost:", best_cost[goal])
print("\nNote: UCS selects minimum accumulated cost, not fewest edges.")
print("  Via I-8+F-8: 4+5+3 = 12  (rejected – higher cost)")
print("  Via H-8    : 2+6   =  8  (selected – lower cost)")

In [ ]:
# ===========================================================================
# TASK 4  NetworkX VISUALIZATION
#   Weighted directed graph with edge costs displayed.
# ===========================================================================
G = nx.DiGraph()
edge_weight_labels = {}
for node in graph:
    for neighbor, cost in graph[node]:
        G.add_edge(node, neighbor, weight=cost)
        edge_weight_labels[(node, neighbor)] = str(cost)

pos = {
    "Islamabad Delivery Hub":  (-3.0,  0.0),
    "I-8 Markaz":              (-1.0,  1.5),
    "H-8 Markaz":              (-1.0, -1.5),
    "F-8 Markaz":              ( 1.0,  1.5),
    "Blue Area Delivery Point": ( 3.0,  0.0),
}

path_set   = set(path)
path_edges = list(zip(path, path[1:]))

node_colors = []
for node in G.nodes():
    if node == start:       node_colors.append("limegreen")
    elif node == goal:      node_colors.append("tomato")
    elif node in path_set:  node_colors.append("gold")
    else:                   node_colors.append("lightblue")

non_path_edges = [e for e in G.edges() if e not in path_edges]

plt.figure(figsize=(13, 6))

nx.draw_networkx_nodes(G, pos, node_color=node_colors, node_size=2400)
nx.draw_networkx_labels(G, pos, font_size=7, font_weight="bold")

nx.draw_networkx_edges(G, pos, edgelist=non_path_edges,
                       arrows=True, arrowsize=20, edge_color="grey",
                       width=1.5, connectionstyle="arc3,rad=0.12")

nx.draw_networkx_edges(G, pos, edgelist=path_edges,
                       arrows=True, arrowsize=25, edge_color="red",
                       width=4, connectionstyle="arc3,rad=0.12")

# Display edge costs on the graph
nx.draw_networkx_edge_labels(G, pos, edge_labels=edge_weight_labels,
                              font_size=10, font_color="darkblue",
                              font_weight="bold")

plt.title(
    f"Task 4 \u2013 UCS: Minimum-Cost Delivery Route  |  Total Cost = {best_cost[goal]}\n"
    "Green=Start | Red=Goal | Gold=UCS Path | Blue numbers=Edge costs",
    fontsize=10
)
plt.axis("off")
plt.tight_layout()
plt.show()